In [5]:
import re, string, json, os, numpy as np
from collections import Counter
from rouge import Rouge

def normalize_answer(s):
    def remove_articles(text):
        return re.sub(r"\b(a|an|the)\b", " ", text)
    def white_space_fix(text):
        return " ".join(text.split())
    def remove_punc(text):
        return "".join(ch for ch in text if ch not in set(string.punctuation))
    def lower(text):
        return text.lower()
    return white_space_fix(remove_articles(remove_punc(lower(s))))

def f1_score(prediction, ground_truth):
    common = Counter(prediction) & Counter(ground_truth)
    num_same = sum(common.values())
    if num_same == 0: return 0
    precision = 1.0 * num_same / len(prediction)
    recall = 1.0 * num_same / len(ground_truth)
    return (2 * precision * recall) / (precision + recall)

def qa_f1_score(prediction, ground_truth, **kwargs):
    return f1_score(normalize_answer(prediction).split(), normalize_answer(ground_truth).split())

def rouge_score(prediction, ground_truth, **kwargs):
    try:
        return Rouge().get_scores([prediction], [ground_truth], avg=True)["rouge-l"]["f"]
    except:
        return 0.0

dataset2metric = {
    "narrativeqa": qa_f1_score, "qasper": qa_f1_score,
    "multifieldqa_en": qa_f1_score, "hotpotqa": qa_f1_score,
    "2wikimqa": qa_f1_score, "musique": qa_f1_score,
    "gov_report": rouge_score, "qmsum": rouge_score,
    "multi_news": rouge_score, "samsum": rouge_score,
}

def score_dataset(jsonl_path):
    with open(jsonl_path) as f:
        data = [json.loads(line) for line in f]
    dataset = os.path.splitext(os.path.basename(jsonl_path))[0]
    metric = dataset2metric.get(dataset, qa_f1_score)
    scores = []
    for item in data:
        best = 0
        for gt in item["answers"]:
            best = max(best, metric(item["pred"], gt))
        scores.append(best)
    return dataset, round(100 * np.mean(scores), 2)



In [12]:
import os

vanilla_dir = "pred/vanilla_run/"
adakv_dir = "pred/adakv_run/"

results = []
for fname in os.listdir(vanilla_dir):
    if not fname.endswith(".jsonl"):
        continue
    
    ds = fname.replace(".jsonl", "")
    van_path = os.path.join(vanilla_dir, fname)
    ada_path = os.path.join(adakv_dir, fname)
    
    _, van_score = score_dataset(van_path)
    
    if os.path.exists(ada_path):
        _, ada_score = score_dataset(ada_path)
        delta = ada_score - van_score
    else:
        ada_score = None
        delta = None
    
    results.append((ds, van_score, ada_score, delta))

# Print comparison table
print(f"{'Dataset':<20} {'Vanilla':>8} {'AdaKV':>8} {'Δ':>8}")
print("-" * 48)
van_scores = []
for ds, v, a, d in sorted(results, key=lambda x: x[3] if x[3] is not None else 0):
    van_scores.append(v)
    a_str = f"{a:>8.2f}" if a is not None else "   N/A  "
    d_str = f"{d:>+8.2f}" if d is not None else "   N/A  "
    print(f"{ds:<20} {v:>8.2f} {a_str} {d_str}")

print("-" * 48)
avg_v = np.mean(van_scores)
completed = [r for r in results if r[2] is not None]
avg_a = np.mean([r[2] for r in completed]) if completed else 0
print(f"{'AVERAGE':<20} {avg_v:>8.2f} {'N/A':>8}" if not completed else
      f"{'AVERAGE':<20} {avg_v:>8.2f} {avg_a:>8.2f} {avg_a-avg_v:>+8.2f}")

Dataset               Vanilla    AdaKV        Δ
------------------------------------------------
gov_report              28.09    14.43   -13.66
multi_news               4.94     2.33    -2.61
qasper                  12.51    11.53    -0.98
qasper_preds            12.51    11.53    -0.98
narrativeqa             18.77    18.13    -0.64
hotpotqa                 8.35     8.17    -0.18
samsum                  30.08    34.22    +4.14
------------------------------------------------
AVERAGE                 16.46    14.33    -2.13
